<a href="https://colab.research.google.com/github/yajima-yasutoshi/ADS2026/blob/main/20260929/%E5%9F%BA%E7%A4%8E%E9%9B%86%E8%A8%88%E3%81%AE%E6%BC%94%E7%BF%92%EF%BC%88marketing_campaign%EF%BC%89.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
#演習

##準備

In [ ]:
%%capture
# Matplotlibで日本語を表示するためのライブラリをインストール
!pip install japanize-matplotlib

In [ ]:
import os
import glob

# 分析に必要なライブラリを読み込む
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
import scipy.stats as st

#日本語表示のためにインポート
import japanize_matplotlib

##データセットの入手

本演習では、以下の Kaggleで公開されている **Customer Personality Analysis** データを使用する。

1. Kaggle の
https://www.kaggle.com/datasets/imakash3011/customer-personality-analysis
を開き、必要ならサインインした後に Download からZIPファイルを取得する。

1. ZIPファイルを展開し、 marketing_campaign.csv を入手する。

> CSVファイルの提供元と利用条件はKaggleのページを確認のこと。

##データの概要

このデータは、ある企業の顧客について、**顧客属性、世帯属性、購買金額、購入チャネル、キャンペーンへの反応**などを記録したデータである。
顧客の特徴や購買行動を理解し、適切な顧客層に適切な商品・販促施策を実施ための分析を想定する。

##データ項目

主な項目は次のとおりである。

### 顧客・世帯属性

| 列名 | 説明 |
|---|---|
| `ID` | 顧客を識別するID |
| `Year_Birth` | 生年 |
| `Education` | 教育レベル |
| `Marital_Status` | 婚姻・パートナー状況 |
| `Income` | 世帯の年間所得 |
| `Kidhome` | 世帯内の子供の人数 |
| `Teenhome` | 世帯内の10代の子供の人数 |
| `Dt_Customer` | 顧客として登録された日 |
| `Recency` | 最後の購入からの経過日数 |
| `Complain` | 過去2年間に苦情があれば1、なければ0 |

### 商品別購入金額（過去2年間）

| 列名 | 説明 |
|---|---|
| `MntWines` | ワインの購入金額 |
| `MntFruits` | 果物の購入金額 |
| `MntMeatProducts` | 肉製品の購入金額 |
| `MntFishProducts` | 魚製品の購入金額 |
| `MntSweetProducts` | 菓子類の購入金額 |
| `MntGoldProds` | Gold productsの購入金額 |

**注意：** 金額の具体的な通貨単位はKaggleのデータカードでは明示されていない。

### 購入・チャネル

| 列名 | 説明 |
|---|---|
| `NumDealsPurchases` | 値引きを利用した購入回数 |
| `NumWebPurchases` | Webサイトでの購入回数 |
| `NumCatalogPurchases` | カタログを使った購入回数 |
| `NumStorePurchases` | 店舗での購入回数 |
| `NumWebVisitsMonth` | 直近1か月のWebサイト訪問回数 |

### キャンペーン

| 列名 | 説明 |
|---|---|
| `AcceptedCmp1` | 第1回キャンペーンを受諾した場合1 |
| `AcceptedCmp2` | 第2回キャンペーンを受諾した場合1 |
| `AcceptedCmp3` | 第3回キャンペーンを受諾した場合1 |
| `AcceptedCmp4` | 第4回キャンペーンを受諾した場合1 |
| `AcceptedCmp5` | 第5回キャンペーンを受諾した場合1 |
| `Response` | 直近のキャンペーンを受諾した場合1 |


##データの読み込み

marketing_campaign.csv が手元のPCに保存されているとする。

以下のセルを実行して、`data_raw` にデータを読み込む。
データの区切り文字は環境や取得元によって異なる可能性があるため、`sep=None` と `engine='python'` を使用して自動判定する。

In [ ]:
# Google Colabへファイルをアップロードする
from google.colab import files
uploaded = files.upload()
file_name = next(iter(uploaded))
data_raw = pd.read_csv(
        file_name,
        sep=None,
        engine="python",
        encoding="utf-8-sig"
    )
print('読み込み完了:', data_raw.shape)

###インターネット経由で読み込むことも可能である

In [ ]:
#!pip -q install kagglehub
import kagglehub
DATASET_HANDLE = "imakash3011/customer-personality-analysis"
dataset_dir = kagglehub.dataset_download(DATASET_HANDLE)

csv_files = glob.glob(
    os.path.join(dataset_dir, "**", "*.csv"),
    recursive=True
)
preferred = [
    path for path in csv_files if os.path.basename(path).lower() == "marketing_campaign.csv"
]

csv_path = preferred[0] if preferred else csv_files[0]
data_raw = pd.read_csv(
        csv_path,
        sep=None,
        engine="python",
        encoding="utf-8-sig"
    )
print('読み込み完了:', data_raw.shape)
print("ファイル:", csv_path)

元データを `data_raw` として保持し、分析には複製した `data` を使用する。
また、`ID` は数値計算を行う項目ではなく識別子なので、文字列として扱う。

In [ ]:
data = data_raw.copy()
data.columns = data.columns.str.strip()
data["ID"] = data["ID"].astype("string")

##演習課題

以下を実施する。

- データの概要と実施したクレンジング処理

  主に確認すべき観点は次のとおりである。

  - 行数と列数
  - 各列の名称とデータ型
  - 欠損値
  - 重複行
  - カテゴリ表記の揺れ
  - 値の範囲
  - 外れ値

- 直近のキャンペーン（`Response`）での顧客の反応の特徴
  - 複数の切り口で反応の違いを見る


- 次回キャンペーンでの重点顧客の提案